In [1]:
import pandas as pd
import numpy as np
import time

# Import the optimized loaders and limits
from PreliminaryTestingOpt import (
    load_openml_suite, load_aeon_suite,
    MAX_CELLS_PER_DATASET
)

def get_imbalance_ratio(y_train, task):
    """Calculates max class / min class for classification training sets."""
    if task == "regression":
        return np.nan
    try:
        if isinstance(y_train, np.ndarray):
            counts = pd.Series(y_train).value_counts()
        else:
            counts = y_train.value_counts()
            
        if len(counts) == 0: 
            return np.nan
        return round(counts.max() / counts.min(), 4)
    except Exception:
        return np.nan

def get_column_counts(X_train, data_type):
    """Accurately calculates standard and flattened categorical/numerical columns."""
    if data_type == "tabular":
        if isinstance(X_train, pd.DataFrame):
            num_cols = X_train.select_dtypes(include=["number"]).shape[1]
            cat_cols = X_train.select_dtypes(exclude=["number"]).shape[1]
        else:
            # Fallback if somehow already numpy
            num_cols = X_train.shape[1] if X_train.ndim > 1 else 1
            cat_cols = 0
        return num_cols, cat_cols
        
    elif data_type == "ts":
        # Time series data from aeon is purely numerical 
        # We need to account for flattening (channels * time steps)
        if isinstance(X_train, np.ndarray) and X_train.ndim == 3:
            num_cols = X_train.shape[1] * X_train.shape[2]
        elif isinstance(X_train, np.ndarray) and X_train.ndim == 2:
            num_cols = X_train.shape[1]
        elif isinstance(X_train, pd.DataFrame):
            try:
                # If nested dataframe, approximate columns by length of lists
                num_cols = sum([len(X_train.iloc[0, i]) for i in range(X_train.shape[1])])
            except Exception:
                num_cols = X_train.shape[1]
        else:
            num_cols = 1
            
        return num_cols, 0 # Time series generally have 0 categorical cols
        
    return 0, 0

if __name__ == "__main__":
    print(f"Starting Benchmark Audit (Max Cells limit enforced: {MAX_CELLS_PER_DATASET:,})")
    
    # 1. Load Datasets
    # Because PreliminaryTestingOpt drops large ones automatically, we pass limit=-1
    datasets = []
    datasets += load_openml_suite(99, "classification", "OpenML-CC18", limit=-1)
    datasets += load_openml_suite(297, "regression", "OpenML-297", limit=-1)
    datasets += load_openml_suite(334, "classification", "OpenML-334", limit=-1)
    datasets += load_openml_suite(335, "regression", "OpenML-335", limit=-1)
    datasets += load_openml_suite(336, "regression", "OpenML-336", limit=-1)
    datasets += load_aeon_suite("TSC", limit=-1)
    datasets += load_aeon_suite("TSER", limit=-1)
    
    print(f"\nProcessing {len(datasets)} total datasets...")
    
    # 2. Extract Audit Information
    audit_records = []
    for ds in datasets:
        X_train = ds["X_train"]
        X_test = ds["X_test"]
        
        # Row counts
        train_rows = len(X_train)
        test_rows = len(X_test)
        total_rows = train_rows + test_rows
        
        # Column counts
        num_cols, cat_cols = get_column_counts(X_train, ds["data_type"])
        total_cols = num_cols + cat_cols
        
        # Total size footprint
        total_cells = total_rows * total_cols
        
        # Imbalance
        imb_ratio = get_imbalance_ratio(ds["y_train"], ds["task"])
        
        audit_records.append({
            "Suite": ds["suite_name"],
            "Task": ds["task"],
            "Data_Type": ds["data_type"],
            "Dataset": ds["name"],
            "Total_Rows": total_rows,
            "Train_Rows": train_rows,
            "Test_Rows": test_rows,
            "Total_Columns": total_cols,
            "Numerical_Cols": num_cols,
            "Categorical_Cols": cat_cols,
            "Total_Cells": total_cells,
            "Train_Imbalance_Ratio": imb_ratio
        })
        
    # 3. Export Data
    df_audit = pd.DataFrame(audit_records)
    
    # Sort for readability: Suite -> Task -> Total Cells (descending)
    df_audit = df_audit.sort_values(by=["Suite", "Total_Cells"], ascending=[True, False])
    
    df_audit.to_csv("benchmark_cell_audit.csv", index=False)
    
    print(f"Audit complete! Data saved to benchmark_cell_audit.csv")
    print("\nSample Output:")
    print(df_audit[["Suite", "Dataset", "Total_Rows", "Total_Columns", "Train_Imbalance_Ratio"]].head())

Global Seed: 42 | Iterative Seeds: [121958, 671155, 131932, 365838, 259178]
Starting Benchmark Audit (Max Cells limit enforced: 5,000,000)
Loaded OpenML-CC18: kr-vs-kp
Loaded OpenML-CC18: letter
Loaded OpenML-CC18: balance-scale
Loaded OpenML-CC18: mfeat-factors
Loaded OpenML-CC18: mfeat-fourier
Loaded OpenML-CC18: breast-w
Loaded OpenML-CC18: mfeat-karhunen
Loaded OpenML-CC18: mfeat-morphological
Loaded OpenML-CC18: mfeat-zernike
Loaded OpenML-CC18: cmc
Loaded OpenML-CC18: optdigits
Loaded OpenML-CC18: credit-approval
Loaded OpenML-CC18: credit-g
Loaded OpenML-CC18: pendigits
Loaded OpenML-CC18: diabetes
Loaded OpenML-CC18: spambase
Loaded OpenML-CC18: splice
Loaded OpenML-CC18: tic-tac-toe
Loaded OpenML-CC18: vehicle
Loaded OpenML-CC18: electricity
Loaded OpenML-CC18: satimage
Loaded OpenML-CC18: eucalyptus
Loaded OpenML-CC18: sick
Loaded OpenML-CC18: vowel
Loaded OpenML-CC18: isolet
Loaded OpenML-CC18: analcatdata_authorship
Loaded OpenML-CC18: analcatdata_dmft
  -> Skipping mnist_7